In [15]:
import csv
import json
import pandas as pd
import numpy as np
import requests

"""
Writing below code as part of 1.1
"""
#=========================================
#=========CODE for 1.1 starts=============
#=========================================

#Function for converting into int and float
def parse_row(row, int_fields, float_fields):
    """Converts specified dictionary keys to integers and floats."""
    clean_row = row.copy()
    for col_title in int_fields:
      row[col_title] = int(row[col_title])

    for col_title in float_fields:
      row[col_title] = float(row[col_title])

    return row

# Listing down which columns should be convered to int and which to float
int_fields = ['age','fee_inr','prior_no_shows']
float_fields = ['distance_km']
final_data = []

accepted_list = []
rejected_list = []

with open ("/content/appointments.csv", "r", encoding="utf-8", newline="") as file:
  reader = csv.DictReader(file)
  print('Header:', reader.fieldnames)
  print()

#Code for printing number of rows
  row_count = list(reader)
  print(f'Number of rows are: {len(row_count)}')


  for row in row_count:
    try:
      converted_row = parse_row(row, int_fields, float_fields)
      accepted_list.append(converted_row)
    except (ValueError, KeyError, TypeError) as e:
      # Optional: store the error reason alongside the rejected row
      rejected_list.append({'row': row, 'error': str(e)})

print(f'\nAccepted ({len(accepted_list)}):', accepted_list)
print(f'\nRejected ({len(rejected_list)}):', rejected_list)
print()

with open('/content/rejected_rows.json', 'w', encoding='utf-8') as json_file:
  json.dump(rejected_list, json_file, indent=2, ensure_ascii=False)

#=========================================
#=========CODE for 1.1 ends===============
#=========================================


#=========================================
#=========CODE for 1.2 starts=============
#=========================================

# ── STEP 1: Load ────────────────────────────────────────────────
df = pd.read_csv("/content/appointments.csv")   # or pd.read_excel(...)

# ── STEP 2: Dimensions ──────────────────────────────────────────
print("Shape:", df.shape)
print()

# ── STEP 3: Structural inspection ──────────────────────────────-
df.info()
print()

# ── STEP 4: Missing value count ────────────────────────────────-
print(f"The number of null values are \n{df.isnull().sum()}")
print()

# ── STEP 5: Total number of duplicated rows ────────────────────────────────-

#duplicates = df[df.duplicated()]
#print("Duplicate rows:")
#print(duplicates)
print(f"The number of duplicated values are {df.duplicated().sum()}")
print()

#------Cleaning up the data-----------
#------Task A Dropping the duplicates
df = df.drop_duplicates()
#print(df_new)
print(f"\nThe number of rows after removing duplicates are {len(df)}")
#------------------------------------

#-----Task B Cleaning booking channel--------
df['booking_channel'] = df['booking_channel'].str.strip().str.title()
#print(df)
print(f"\nThe number of rows after cleaning booking channel are {len(df)}")
#------------------------------------

#----------Task C Convert to datetime--------
df["appt_date"] = pd.to_datetime(df["appt_date"], format="%d-%m-%Y", errors="coerce")
df["booked_on"] = pd.to_datetime(df["booked_on"], format="%d-%m-%Y", errors="coerce")


missing_date = df[df['appt_date'].isna()]
print()
print(f"Missing dates are \n{missing_date}")
df = df.dropna(subset=['appt_date'])
#print(df)

print()
missing_status = df[df['status'].isna()]
print(f"Missing status are \n{missing_status}")

#------Task D Drop missing dates and status-------
df = df.dropna(subset=['appt_date'])
#print(df)
df = df.dropna(subset=['status'])
print()
print(f"The new dataframe is \n{df}")

print(f"\nThe number of rows after 1.2 are {len(df)}")

#=========================================
#=========CODE for 1.2 ends===============
#=========================================

#=========================================
#=========CODE for 1.3 starts=============
#=========================================
print()
#df['age'] = df['age'].where(df['age'].between(1, 100))
#print(df['age'])

invalid_age = (~df['age'].between(1, 100) & df['age'].notna()).sum()
df['age'] = df['age'].where(df['age'].between(1, 100))

print(f"The number of invalid ages are {invalid_age}")
#print(df['age'])

#1.3b
df.fillna({'age': df['age'].median()}, inplace=True)
#df['age'].fillna(df['age'].median(), inplace=True)
print(df['age'])
df['age'] = df['age'].round().astype(int)

#df.fillna({df['distance_km'].median}, inplace=True)
df.fillna({'distance_km': df['distance_km'].median()}, inplace=True)
#df['distance_km'].fillna(df['distance_km'].median(), inplace=True)
print(df['distance_km'])

#=====================================================
#1.3c
#=====================================================


# FOR DISTANCE_KM
#=====================================================

Q1 = df['distance_km'].quantile(0.25)
Q3 = df['distance_km'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
print(f"\nLower bound for distance_km is {lower_bound} and Upper bound for distance_km is {upper_bound}")

# Count how many values changed (excluding NaNs)
capped_distance_km = ((df['distance_km'] < lower_bound) | (df['distance_km'] > upper_bound)).sum()
print(f"Number of capped values for Distance_km are {capped_distance_km}")

df['distance_km'] = np.clip(df['distance_km'], lower_bound, upper_bound)


# FOR FEE_INR
#=====================================================

Q1 = df['fee_inr'].quantile(0.25)
Q3 = df['fee_inr'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
print(f"\nLower bound for fee_inr is {lower_bound} and Upper bound for fee_inr is {upper_bound}")

capped_fee_inr = ((df['fee_inr'] < lower_bound) | (df['fee_inr'] > upper_bound)).sum()
print(f"Number of capped values for Fee_inr are {capped_fee_inr}")

df['fee_inr'] = np.clip(df['fee_inr'], lower_bound, upper_bound)

#1.3d
#=====================================================

df['distance_km'] = df['distance_km'].round(1)
df['fee_inr'] = df['fee_inr'].round().astype(int)


#********************************************************************
#1.4 CODE STARTS
#********************************************************************
print()
df["lead_days"] = (df["appt_date"] - df["booked_on"]).dt.days
print(df)

df["no_show"] = (df['status'] == 'No-Show')
df["no_show"] = df["no_show"].astype(int)
print(df)

X = df[["age", "distance_km","fee_inr","lead_days"]].to_numpy()

# Average sales per product
X_avg = np.mean(X, axis=0)
print("Average sales per product:")
print(X_avg)

mask = X[:, 3] > 14
#print(mask)
#total_filtered_lead_days = X[mask, 3].sum()
print(mask.sum())

print(mask.mean())


#1.4 c CODE STARTS
#********************************************************************

z_distance_km = X[:, 1]

# Compute z-scores
z_scores = (z_distance_km - np.mean(z_distance_km)) / np.std(z_distance_km)
#print(z_scores)
pos_z_scores = np.abs(z_scores)
z_normal_score = pos_z_scores > 2
print(f"{z_normal_score.sum()} rows have |z| > 2 for distance_km")
#print(z_normal_score)

#********************************************************************
#1.5 CODE STARTS
#********************************************************************
collected_temps = []
url = 'https://api.open-meteo.com/v1/forecast'
params = {
    'latitude': '18.5074',
    'longitude': '73.8077',
    'hourly': 'temperature_2m',
    'forecast_days': 1
}
try:
    response = requests.get(url, params=params, timeout =10)

    # Check status code
    if response.status_code == 200:
        data = response.json()
        #print(data)
        temp_c = data['hourly']['temperature_2m']
        #print(f"\ntemp_c: {temp_c}")
        first_value = temp_c[0] if temp_c else None
        collected_temps.append(first_value)
        #new_df = pd.DataFrame({"temp_c": [[first_value]]})
        #print(collected_temps)
        #print(df)
    else:
        print(f"Error: Status code {response.status_code}")
except requests.exceptions.RequestException as e:
    print(f"Error: {e}")




#=======================================================
#=======2nd clinic================

url = 'https://api.open-meteo.com/v1/forecast'
params = {
    'latitude': '18.5089',
    'longitude': '73.926',
    'hourly': 'temperature_2m',
    'forecast_days': 1
}
try:
    response = requests.get(url, params=params, timeout =10)

    # Check status code
    if response.status_code == 200:
        data = response.json()
        #print(data)
        temp_c = data['hourly']['temperature_2m']
        #print(f"\ntemp_c: {temp_c}")
        first_value = temp_c[0] if temp_c else None
        collected_temps.append(first_value)
        #new_df = pd.DataFrame({"temp_c": [[first_value]]})
        #print(collected_temps)
    else:
        print(f"Error: Status code {response.status_code}")
except requests.exceptions.RequestException as e:
    print(f"Error: {e}")

#==========3rd clinic==============
url = 'https://api.open-meteo.com/v1/forecast'
params = {
    'latitude': '21.1458',
    'longitude': '79.0882',
    'hourly': 'temperature_2m',
    'forecast_days': 1
}
try:
    response = requests.get(url, params=params, timeout =10)

    # Check status code
    if response.status_code == 200:
        data = response.json()
        #print(data)
        temp_c = data['hourly']['temperature_2m']
        #print(f"\ntemp_c: {temp_c}")
        first_value = temp_c[0] if temp_c else None
        collected_temps.append(first_value)
        #new_df = pd.DataFrame({"temp_c": [[first_value]]})
        #print(collected_temps)
    else:
        print(f"Error: Status code {response.status_code}")
except requests.exceptions.RequestException as e:
    print(f"Error: {e}")

#==========4th clinic==============
url = 'https://api.open-meteo.com/v1/forecast'
params = {
    'latitude': '26.8553',
    'longitude': '75.815',
    'hourly': 'temperature_2m',
    'forecast_days': 1
}
try:
    response = requests.get(url, params=params, timeout =10)

    # Check status code
    if response.status_code == 200:
        data = response.json()
        #print(data)
        temp_c = data['hourly']['temperature_2m']
        #print(f"\ntemp_c: {temp_c}")
        first_value = temp_c[0] if temp_c else None
        collected_temps.append(first_value)
        #new_df = pd.DataFrame({"temp_c": [[first_value]]})
        #print(collected_temps)
    else:
        print(f"Error: Status code {response.status_code}")
except requests.exceptions.RequestException as e:
    print(f"Error: {e}")

#==========5th clinic==============
url = 'https://api.open-meteo.com/v1/forecast'
params = {
    'latitude': '26.9124',
    'longitude': '75.7873',
    'hourly': 'temperature_2m',
    'forecast_days': 1
}
try:
    response = requests.get(url, params=params, timeout =10)

    # Check status code
    if response.status_code == 200:
        data = response.json()
        #print(data)
        temp_c = data['hourly']['temperature_2m']
        #print(f"\ntemp_c: {temp_c}")
        first_value = temp_c[0] if temp_c else None
        collected_temps.append(first_value)
        #new_df = pd.DataFrame({"temp_c": [[first_value]]})
        #print(collected_temps)
    else:
        print(f"Error: Status code {response.status_code}")
except requests.exceptions.RequestException as e:
    print(f"Error: {e}")

#==========6th clinic==============
url = 'https://api.open-meteo.com/v1/forecast'
params = {
    'latitude': '10.0261',
    'longitude': '76.3085',
    'hourly': 'temperature_2m',
    'forecast_days': 1
}
try:
    response = requests.get(url, params=params, timeout =10)

    # Check status code
    if response.status_code == 200:
        data = response.json()
        #print(data)
        temp_c = data['hourly']['temperature_2m']
        #print(f"\ntemp_c: {temp_c}")
        first_value = temp_c[0] if temp_c else None
        collected_temps.append(first_value)
        #new_df = pd.DataFrame({"temp_c": [[first_value]]})
        #print(collected_temps)
    else:
        print(f"Error: Status code {response.status_code}")
except requests.exceptions.RequestException as e:
    print(f"Error: {e}")

#==========7th clinic==============
url = 'https://api.open-meteo.com/v1/forecast'
params = {
    'latitude': '22.7533',
    'longitude': '75.8937',
    'hourly': 'temperature_2m',
    'forecast_days': 1
}
try:
    response = requests.get(url, params=params, timeout =10)

    # Check status code
    if response.status_code == 200:
        data = response.json()
        #print(data)
        temp_c = data['hourly']['temperature_2m']
        #print(f"\ntemp_c: {temp_c}")
        first_value = temp_c[0] if temp_c else None
        collected_temps.append(first_value)
        #new_df = pd.DataFrame({"temp_c": [[first_value]]})
        #print(collected_temps)
    else:
        print(f"Error: Status code {response.status_code}")
except requests.exceptions.RequestException as e:
    print(f"Error: {e}")

#==========8th clinic==============
url = 'https://api.open-meteo.com/v1/forecast'
params = {
    'latitude': '26.8467',
    'longitude': '80.9462',
    'hourly': 'temperature_2m',
    'forecast_days': 1
}
try:
    response = requests.get(url, params=params, timeout =10)

    # Check status code
    if response.status_code == 200:
        data = response.json()
        #print(data)
        temp_c = data['hourly']['temperature_2m']
        #print(f"\ntemp_c: {temp_c}")
        first_value = temp_c[0] if temp_c else None
        collected_temps.append(first_value)
        #new_df = pd.DataFrame({"temp_c": [[first_value]]})
        #print(collected_temps)
    else:
        print(f"Error: Status code {response.status_code}")
except requests.exceptions.RequestException as e:
    print(f"Error: {e}")


latest_df = pd.DataFrame({'temp_c': collected_temps})
print('\nNew DataFrame:')
print(latest_df)
#latest_df = pd.DataFrame({"Column_Name": [new_df]})
#print(latest_df)
existing_df = pd.read_csv('/content/clinics.csv')

# Assign the new column (requires existing_df to have 8 rows)
existing_df['temp_c'] = latest_df['temp_c']

# Save the updated DataFrame back to the CSV without the pandas index
existing_df.to_csv('/content/clinics_enriched.csv', index=False)

#********************************************************************
#1.6 CODE STARTS
#********************************************************************

df_sorted = df.sort_values(by='appointment_id')
df["booked_on"] = pd.to_datetime(df["booked_on"]).dt.strftime("%Y-%m-%d")
df["appt_date"] = pd.to_datetime(df["appt_date"]).dt.strftime("%Y-%m-%d")

df.to_csv('/content/appointments_clean.csv', index=False)

# 1. Convert to datetime first so sorting works chronologically
#df["booked_on"] = pd.to_datetime(df["booked_on"])
#df["appt_date"] = pd.to_datetime(df["appt_date"])

# 2. Sort by date (and optionally appointment_id as a tie-breaker)
#df_sorted = df.sort_values(by=["appt_date", "appointment_id"], ascending=True)

# 3. Format dates as YYYY-MM-DD strings after sorting
#df_sorted["booked_on"] = df_sorted["booked_on"].dt.strftime("%Y-%m-%d")
#df_sorted["appt_date"] = df_sorted["appt_date"].dt.strftime("%Y-%m-%d")

# 4. Save the sorted DataFrame (df_sorted), not the unsorted original (df)
#df_sorted.to_csv("/content/new_appointments_clean.csv", index=False)


csv_file = '/content/clinics_enriched.csv'       # Your input CSV file
sql_file = '/content/load_data.sql'     # Your generated SQL file
#table_name = 'clinics'   # Target database table name


def csv_to_sql(csv_path, table_name, columns, output_path, mode="w"):
    with open(csv_path, mode="r", encoding="utf-8") as csv_file, \
         open(output_path, mode=mode, encoding="utf-8") as sql_file:

        reader = csv.DictReader(csv_file)
        cols_str = ", ".join(columns)

        for row in reader:
            # Format each value: wrap strings in single quotes and escape internal quotes
            values = []
            for col in columns:
                val = row[col].replace("'", "''")
                values.append(f"'{val}'" if val != "" else "NULL")

            vals_str = ", ".join(values)
            sql_file.write(f"INSERT INTO {table_name} ({cols_str}) VALUES ({vals_str});\n")

# Step 1: Create output.sql and add first table's data ('w' overwrites/creates file)
csv_to_sql("/content/clinics_enriched.csv", "clinics", ["clinic_id", "clinic_name", "city","tier","latitude","longitude","temp_c"], "/content/load_data.sql", mode="w")

# Step 2: Add second table's data to the same file ('a' appends to the end)
csv_to_sql("/content/appointments_clean.csv", "appointments", ["appointment_id", "patient_id","clinic_id","booked_on","appt_date","age","gender","distance_km","fee_inr","booking_channel","visit_type","reminder_sent","prior_no_shows","status","lead_days","no_show"], "/content/load_data.sql", mode="a")





Header: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']

Number of rows are: 1020

Accepted (988): [{'appointment_id': 'A0001', 'patient_id': 'P509', 'clinic_id': 'C05', 'booked_on': '24-01-2025', 'appt_date': '25-01-2025', 'age': 60, 'gender': 'F', 'distance_km': 18.0, 'fee_inr': 850, 'booking_channel': 'Phone', 'visit_type': 'Consultation', 'reminder_sent': 'Yes', 'prior_no_shows': 1, 'status': 'No-Show'}, {'appointment_id': 'A0002', 'patient_id': 'P046', 'clinic_id': 'C07', 'booked_on': '27-05-2025', 'appt_date': '06-06-2025', 'age': 39, 'gender': 'M', 'distance_km': 12.5, 'fee_inr': 350, 'booking_channel': 'Walk-in', 'visit_type': 'Follow-up', 'reminder_sent': 'No', 'prior_no_shows': 2, 'status': 'Attended'}, {'appointment_id': 'A0003', 'patient_id': 'P009', 'clinic_id': 'C02', 'booked_on': '24-03-2025', 'appt_date': '30-03-2025', 'age': 6

In [ ]:
from google.colab import drive
drive.mount('/content/drive')